# 강의 01 · 실습 12 — 첫 API 서비스 · (2-2) 빈칸 채우기 II

## 1. 문제상황

- 팀에서 만든 질문 답변 프로그램은 파이썬 파일 안의 함수로만 존재합니다.
- 동료가 답변을 받으려면 파이썬을 설치하고 파일을 받아 함수를 직접 불러야 합니다.
- 질문을 보내는 방법이 사람마다 다르고, 프로그램이 바뀔 때마다 파일을 다시 나눠 주어야 합니다.
- 동료가 늘어나면 설치 안내와 파일 배포를 사람이 그만큼 반복해야 합니다.

## 2. 문제와 목표

- **문제**: 답변 함수가 파이썬 파일 안에만 있습니다. 파이썬을 모르는 동료는 함수를 부를 수 없고, 배포도 사람이 반복합니다.
- **목표**: 질문을 받아 답변을 돌려주는 함수를 만들고, 그 함수에 주소를 붙여 개발 서버를 켜고, 터미널에서 주소를 불러 답변이 돌아오는 서비스를 만듭니다.
    - 함수: 질문 문자열을 받아 모델을 한 번 호출하고 답변 문자열을 돌려주는 `ask`.
    - 주소: 질문을 받는 `/ask`(POST)와 서버가 켜졌는지 보는 `/healthz`(GET).
    - 개발 서버: `fastapi dev`로 켜는 서버. 노트북 안에서는 8012번 포트, 터미널에서는 기본 8000번 포트.
- **목표 달성 여부의 판정 기준**: 서버가 켜진 상태에서 질문을 담은 JSON을 보냈을 때, 답변 문장이 든 JSON이 돌아오는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec01_ex12_s1_diagram.svg)

## 4. 단계별 요구사항

1. **요청의 모양을 선언합니다.**
    - 서비스가 받을 JSON은 `question` 칸 하나를 가집니다.
    - `BaseModel`을 상속한 `AskRequest` 클래스로 선언합니다.
    - 응답은 `answer` 칸을 가진 딕셔너리로 돌려줍니다.
2. **답변 함수를 만듭니다.**
    - `ask` 함수는 질문 문자열을 받아 모델을 한 번 호출하고 답변 문자열을 돌려줍니다.
    - 모델 이름은 인자로 빼고 기본값을 둡니다.
    - 웹과 무관하게 노트북에서 직접 불러 동작을 확인합니다.
3. **앱을 만들고 주소를 붙입니다.**
    - `FastAPI()`로 앱을 만들고, `/ask` 주소에 POST 요청이 오면 `ask` 함수를 부르고 `{"answer": 답변}`을 돌려주는 함수를 등록합니다.
    - 서버가 켜졌는지 모델 호출 없이 확인하는 `/healthz` 주소도 둡니다.
4. **파일로 모아 서버를 켜고 호출합니다.**
    - 요구사항 1~3의 코드를 `lec01_ex12_app.py` 파일 하나에 모으고, `fastapi dev`로 개발 서버를 켠 뒤, 질문 JSON을 `/ask`에 보내 답변 JSON을 받습니다.
    - 준비 코드는 「[healthz] 서버 준비 완료」「[상태] 200」「[응답] …」「[서버] 종료」 줄을 차례로 출력합니다.

## 5. 코드 골격 — FastAPI 서비스 4단

FastAPI로 서비스를 세우는 순서는 다음 네 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 네 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 입출력 모양 선언 | 서비스가 받을 값과 돌려줄 값의 모양을 클래스로 선언합니다 | `class AskRequest(BaseModel)` | 1 |
| ② 처리 함수 구현 | 값을 받아 결과를 돌려주는 함수를 웹과 무관하게 먼저 만듭니다 | `def ask(question) -> str` | 2 |
| ③ 앱 생성·엔드포인트 등록 | 앱을 만들고, 주소와 함수를 데코레이터로 잇습니다 | `app = FastAPI()`, `@app.post("/ask")` | 3 |
| ④ 기동·호출 확인 | 개발 서버를 띄우고 주소를 불러 답이 돌아오는지 확인합니다 | `fastapi dev lec01_ex12_app.py` → `/ask` | 4 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 확인합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- `subprocess`·`sys`·`time`·`httpx`는 단계 ④에서 노트북 안에서 서버를 켜고 부르는 데 씁니다.

In [ ]:
# 여기에 단계 0(라이브러리 불러오기, .env 읽기, 모델 이름 준비)을 작성합니다.

### 단계 ① — 입출력 모양 선언 (요구사항 1)

서비스가 받을 요청 본문의 모양을 클래스로 선언합니다. `BaseModel`을 상속한 클래스의 칸 이름과 타입이 요청 JSON의 계약입니다. `question` 칸이 없거나 문자열이 아닌 요청은 앱이 함수를 부르기 전에 거절합니다. 응답은 `answer` 칸을 가진 딕셔너리이며 별도 클래스 없이 돌려줍니다.

In [ ]:
# 여기에 단계 ①(요청 모양 AskRequest 선언)을 작성합니다.

### 단계 ② — 처리 함수 구현 (요구사항 2)

값을 받아 결과를 돌려주는 함수를 웹과 무관하게 먼저 만듭니다.

- `completion`은 모델 이름과 메시지 목록을 받아 모델을 한 번 호출합니다. 모델 이름 문자열만 바꾸면 다른 모델로 교체됩니다.
- 함수 머리의 타입힌트가 무엇을 받아 무엇을 돌려주는지를 적습니다. 설명문(docstring)은 함수가 하는 일을 적습니다.
- 아래 셀은 함수를 만든 뒤 노트북에서 직접 한 번 불러 봅니다. 서버 없이 함수만으로 답변이 돌아오면 다음 단계로 갑니다.

In [ ]:
# 여기에 단계 ②(처리 함수 ask 구현과 직접 호출)를 작성합니다.

### 단계 ③ — 앱 생성·엔드포인트 등록 (요구사항 3)

`FastAPI()`가 서비스의 몸체입니다. `@app.post("/ask")` 데코레이터가 `/ask` 주소로 온 POST 요청을 바로 아래 함수에 잇습니다.

- 함수의 인자 타입을 `AskRequest`로 적으면, 앱이 요청 JSON을 그 모양으로 검사해 넘겨줍니다.
- 함수는 `ask`를 부른 결과를 `answer` 칸에 담아 돌려줍니다. 앱이 딕셔너리를 JSON으로 바꿉니다.
- `/healthz`는 서버가 켜졌는지 확인하는 주소입니다. 모델을 호출하지 않으므로 비용 없이 부를 수 있습니다.

In [ ]:
# 여기에 단계 ③(앱 생성, /healthz 와 /ask 엔드포인트 등록)을 작성합니다.

### 단계 ④ — 기동·호출 확인 (요구사항 4)

서버는 노트북 셀이 아니라 파일을 읽어 켜집니다. 그래서 단계 ①~③의 코드를 파일 하나에 모읍니다.

- 첫 셀은 `%%writefile`로 `lec01_ex12_app.py` 파일을 씁니다. 파일 내용은 단계 ①~③과 같습니다.
- 둘째 셀은 노트북 안에서 개발 서버를 켜고, `/healthz`로 준비를 기다린 뒤, `/ask`에 질문을 보내고, 서버를 끕니다. 포트는 8012로 고정합니다.
- 터미널에서 직접 켜고 부르는 명령은 「7. 실행 결과 확인」에 있습니다. 실무에서는 터미널 방식으로 켭니다.

In [ ]:
# 여기에 단계 ④의 파일 모으기(첫 줄 %%writefile lec01_ex12_app.py, 그 아래 단계 ①~③ 코드)를 작성합니다.

In [ ]:
PORT = 8012
BASE = f"http://127.0.0.1:{PORT}"

# 노트북 안에서 개발 서버를 켠다. 터미널의 `fastapi dev lec01_ex12_app.py` 와 같은 명령이다.
env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
server = subprocess.Popen(
    [sys.executable, "-m", "fastapi", "dev", "lec01_ex12_app.py", "--port", str(PORT), "--no-reload"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env,
)
try:
    # 서버가 준비될 때까지 /healthz 를 최대 30번 두드린다.
    for attempt in range(30):
        try:
            if httpx.get(f"{BASE}/healthz", timeout=2).status_code == 200:
                break
        except httpx.HTTPError:
            time.sleep(1)
    else:
        raise RuntimeError("서버가 30초 안에 준비되지 않았습니다.")
    print("[healthz] 서버 준비 완료")

    body = {"question": "FastAPI가 무엇인지 한 문장으로 답해 주세요."}
    response = httpx.post(f"{BASE}/ask", json=body, timeout=120)
    print("[요청]", body)
    print("[상태]", response.status_code)
    print("[응답]", response.json())
finally:
    server.terminate()
    server.wait(timeout=10)
    print("[서버] 종료")

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 단계 ② 셀에서 `ask` 함수가 답변 문장을 출력합니다. 서버 없이 함수만으로 모델 호출이 성립했다는 뜻입니다.
2. 단계 ④ 둘째 셀에서 `[healthz] 서버 준비 완료`가 찍힌 뒤 `[상태] 200`과 `answer` 칸이 든 `[응답]`이 찍힙니다. 서버 기동과 호출이 성립했다는 뜻입니다.
3. 마지막 줄에 `[서버] 종료`가 찍힙니다. 노트북이 켠 서버는 노트북이 끕니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다.

터미널에서 직접 켜고 부를 때는 다음 두 명령을 씁니다. 첫 터미널에서 서버를 켜고, 둘째 터미널에서 질문을 보냅니다. 질문은 같은 폴더의 `body_ko.json` 파일에 담겨 있습니다.

```
fastapi dev lec01_ex12_app.py
```

```
curl -X POST http://127.0.0.1:8000/ask -H "Content-Type: application/json" --data-binary "@body_ko.json"
```

- 포트가 다릅니다. 노트북의 단계 ④는 노트북이 켜고 끄는 서버라 8012 포트를 쓰고, 터미널의 `fastapi dev`는 기본 포트 8000을 씁니다. 그래서 위 curl 주소는 8000입니다.
- 파워셸의 `curl`은 다른 명령의 별칭이므로 `curl.exe`로 씁니다.
- 한글 질문은 명령줄에 직접 쓰지 않고 파일로 보냅니다. 파일은 UTF-8로 저장합니다.